# Week 5: Full Fine-Tuning — Unleash All 25.6M Parameters
**TMI-102 | PaAC, IIT Roorkee**

---

### What you'll do this week
1. Take last week's model and **UNFREEZE** all layers
2. Use a **lower learning rate** (1e-4 instead of 1e-3)
3. Watch accuracy jump from ~55% → **~97%**!
4. Compare training time: ~146s/epoch vs ~9s/epoch

### Key Insight
When we unfreeze the backbone, the model can adapt its ImageNet features specifically for galaxy morphology.

### Q1 (5 pts)
Why do we use a *lower* learning rate (1e-4) for fine-tuning compared to training from scratch (1e-2)?

> *Hint: we don't want to destroy the useful ImageNet features*

> *We use a lower learning rate (such as 1e-4) for fine-tuning because the pre-trained model (like ResNet50 trained on ImageNet) already possesses valuable, generalized feature extractors (such as edge, texture, and shape detectors) learned from millions of images.If we use a high learning rate (like 1e-2), the large gradient updates will drastically change the pre-trained weights too quickly. This causes catastrophic forgetting, destroying the useful feature representations the network has already built. A lower learning rate allows the model to make gentle, fine-grained adjustments, adapting the pre-trained features specifically to our new dataset (like Galaxy Zoo) without erasing what it previously learned.*

In [1]:
# ── Setup (same as Week 4) ──
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import numpy as np, matplotlib.pyplot as plt, time

DATASET_DIR = r'C:\Users\Lenovo\Documents\pranay 1\python 1.0\Galaxy_Dataset\dataset_split'
IMG_SIZE = 224; BATCH_SIZE = 32

datagen = ImageDataGenerator(rescale=1./255)
train_gen = datagen.flow_from_directory(f'{DATASET_DIR}/train', target_size=(IMG_SIZE,IMG_SIZE), batch_size=BATCH_SIZE, class_mode='categorical')
val_gen = datagen.flow_from_directory(f'{DATASET_DIR}/val', target_size=(IMG_SIZE,IMG_SIZE), batch_size=BATCH_SIZE, class_mode='categorical')
test_gen = datagen.flow_from_directory(f'{DATASET_DIR}/test', target_size=(IMG_SIZE,IMG_SIZE), batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False)

from tensorflow.keras.preprocessing.image import ImageDataGenerator

# 1. Define your ImageDataGenerator with rescaling
datagen = ImageDataGenerator(rescale=1./255)

# 2. Make sure class_mode is set to 'categorical'
train_gen = datagen.flow_from_directory(
    r'C:\Users\Lenovo\Documents\pranay 1\python 1.0\Galaxy_Dataset\dataset_split\train', 
    target_size=(224, 224), # Update to match your IMG_SIZE variable if it's different
    batch_size=32,          # Update to match your BATCH_SIZE variable if it's different
    class_mode='categorical'
)

val_gen = datagen.flow_from_directory(
    r'C:\Users\Lenovo\Documents\pranay 1\python 1.0\Galaxy_Dataset\dataset_split\val', 
    target_size=(224, 224), 
    batch_size=32, 
    class_mode='categorical'
)

Found 43104 images belonging to 3 classes.
Found 9234 images belonging to 3 classes.
Found 9240 images belonging to 3 classes.
Found 43104 images belonging to 3 classes.
Found 9234 images belonging to 3 classes.


### Task 1: Build & UNFREEZE the model (15 pts)

This is almost identical to Week 4 — but with **one critical change**. Find it!

In [2]:
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))

# ╔══════════════════════════════════════════════╗
# ║  THE BIG CHANGE: set trainable = ???         ║
# ║  (In Week 4 this was False. What should it   ║
# ║   be now to UNFREEZE all layers?)            ║
# ╚══════════════════════════════════════════════╝
for layer in base_model.layers:
    layer.trainable = True   # <-- TODO: True or False?

inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = base_model(inputs, training=True)   # training=True lets BatchNorm update
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(3, activation='softmax')(x)
model = Model(inputs, outputs)

trainable = sum(tf.keras.backend.count_params(w) for w in model.trainable_weights)
print(f"🔓 Trainable parameters: {trainable:,}")
print(f"   (should be ~25.6 million — ALL of ResNet-50!)")

🔓 Trainable parameters: 24,059,907
   (should be ~25.6 million — ALL of ResNet-50!)


### Task 2: Train with lower LR (10 pts)

Notice `learning_rate=1e-4` — 10× lower than last week.

In [3]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),  # Lower LR!
    loss='categorical_crossentropy',
    metrics=['accuracy']
)


start = time.time()
history = model.fit(
    train_gen, 
    validation_data=val_gen, 
    epochs=15,
    callbacks=[
        EarlyStopping(patience=5, restore_best_weights=True),
        ReduceLROnPlateau(patience=3, factor=0.5, verbose=1)
    ]
)
elapsed = time.time() - start
print(f"\n⏱️ Total training time: {elapsed:.0f}s ({elapsed/len(history.history['loss']):.0f}s per epoch)")

Epoch 1/15
  16/1347 ━━━━━━━━━━━━━━━━━━━━ 4:17:16 12s/step - accuracy: 0.6328 - loss: 0.7597

KeyboardInterrupt: 

### Task 3: Evaluate & Compare (15 pts)

In [ ]:
# Plot curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history.history['accuracy'], label='Train')
ax1.plot(history.history['val_accuracy'], label='Val')
ax1.set_title('Full Fine-Tune — Accuracy'); ax1.legend(); ax1.grid(True)
ax2.plot(history.history['loss'], label='Train')
ax2.plot(history.history['val_loss'], label='Val')
ax2.set_title('Full Fine-Tune — Loss'); ax2.legend(); ax2.grid(True)
plt.tight_layout(); plt.show()

# Test
test_loss, test_acc = model.evaluate(test_gen)
print(f"\n🎯 Full Fine-Tune Test Accuracy: {test_acc*100:.2f}%")
print(f"📈 That's a MASSIVE jump from ~55% (Week 4)!")

NameError: name 'plt' is not defined

### Q2 (5 pts)
Fill in this comparison with your actual numbers:

| | Frozen (Week 4) | Full Fine-Tune (This week) |
|:---|:---:|:---:|
| Trainable Params | ~2,050 | ~25,600,000 |
| Test Accuracy | % | % |
| Time per Epoch | s | s |

### Q3 (5 pts)
Was the accuracy improvement worth the ~3,000× increase in parameters and ~15× slower training? When might it NOT be worth it?

> *Your answer here*

---
**Submit as:** `Week_5_YourName.ipynb`